In [ ]:
# =======================================================================
# Setup - run this cell first
# =======================================================================
%matplotlib inline
import os
os.makedirs("figures", exist_ok=True)

# Point this at the folder holding the raw MEA CSV exports.
# The data are confidential and are not included in this repository.
os.environ.setdefault("MEA_INPUT_DIR", "data/raw")
os.environ.setdefault("MEA_MODEL", "models/tier2_rf_model.joblib")
print("input folder:", os.environ["MEA_INPUT_DIR"])


In [ ]:
# =======================================================================
# Import and Preprocessing pipeline code cell
#========================================================================


# ------------------------------------
# Import libraries
# ------------------------------------

import os, glob, re
import pandas as pd


# ------------------------------------
# Configuration
# ------------------------------------

INPUT_DIR = os.environ.get("MEA_INPUT_DIR", "data/raw")   # folder of raw MEA CSV exports
OUTPUT_FILE = "preprocessed_mea.csv"     # the preprocessed table

# Columns to remove: leftover row-number columns, the recording name,
# the always-TRUE "Active" flag, Concemtration(all Nan values) and "Treatment" (same as "Control", so we keep Control).
DROP = ["", "Unnamed: 0", "...1", "Filename", "Active", "Treatment", "Concentration"]

# The columns that together identify one well in one recording.
KEY = ["Plate", "Timepoint_dpp", "Well"]


def load_file(filepath):
    """Read one CSV file and clean it up."""
    df = pd.read_csv(filepath)

    label = str(df["Filename"].iloc[0])

    # remove the junk columns (ignore any that aren't in this file)
    df = df.drop(columns=[c for c in df.columns if str(c).strip() in DROP], errors="ignore")
    df["Well"] = df["Well"].astype(str).str.strip().str.upper()

    # turn the measurement columns into numbers
    for c in df.columns:
        if c not in ["Well", "Control", "Barcode"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # read the plate number and timepoint out of the recording label
    plate = re.search(r"plate[\s_]*(\d+)", label, re.IGNORECASE)
    dpp = re.search(r"(\d+)\s*dpp", label, re.IGNORECASE)
    df["Plate"] = int(plate.group(1)) if plate else None
    df["Timepoint_dpp"] = int(dpp.group(1)) if dpp else None

    return df


# ------------------------------------
# Load and preprocess the Data
# ------------------------------------

# read every file, sorting them into spike and SB groups
spike_tables, sb_tables = [], []
for path in sorted(glob.glob(os.path.join(INPUT_DIR, "*.csv"))):
    name = os.path.basename(path).lower()
    if name.endswith("_spike_stats.csv"):
        spike_tables.append(load_file(path))
    elif name.endswith("_sb_stats.csv"):
        sb_tables.append(load_file(path))

# stack all spike files together, and all SB files together
spike = pd.concat(spike_tables, ignore_index=True)
sb = pd.concat(sb_tables, ignore_index=True)

# from the SB table keep only columns spike doesn't already have
sb_keep = KEY + [c for c in sb.columns if c not in spike.columns and c not in KEY]
sb = sb[sb_keep]

# join spike and SB into one wide table, one row per well
wide = spike.merge(sb, on=KEY, how="outer", validate="one_to_one")

# put the identifier columns first, then sort
front = [c for c in ["Barcode", "Plate", "Timepoint_dpp", "Well", "Control"] if c in wide.columns]
wide = wide[front + [c for c in wide.columns if c not in front]]
wide = wide.sort_values(KEY).reset_index(drop=True)


# ------------------------------------
# Save to CSV
# ------------------------------------

wide.to_csv(OUTPUT_FILE, index=False)
print("Saved", len(wide), "wells to", OUTPUT_FILE)


In [ ]:
# =======================================================================
# Tier_1 quality control code cell
#========================================================================


# QC: flag low-activity wells (non-viable tissue) as fail, everything else pass
low_spikes = wide["TotalSpikesE"] < 50
low_active_elec = wide["ActiveElec"] < 4

# Fail if either condition is met
low_activity = low_spikes | low_active_elec

wide["QC_low_activity"] = low_activity.map({True: "fail", False: "pass"})

wide.to_csv("low_activity_wells_flagged.csv", index=False)
print("Saved", len(wide), "wells to", "low_activity_wells_flagged.csv")


# Overall counts
print(f"{(wide['QC_low_activity'] == 'fail').sum()} wells flagged as fail")
print(f"{(wide['QC_low_activity'] == 'pass').sum()} wells flagged as pass")

# Reason counts
print("\nFailure reasons:")
print(f"Active electrodes < 4 : {low_active_elec.sum()} wells")
print(f"Spikes per electrode < 50 : {low_spikes.sum()} wells")

# Wells failing both criteria
both = (low_active_elec & low_spikes).sum()
print(f"Failed both criteria : {both} wells")

# Wells failing only one criterion
print(f"Failed only ActiveElec criterion : {(low_active_elec & ~low_spikes).sum()} wells")
print(f"Failed only TotalSpikesE criterion : {(low_spikes & ~low_active_elec).sum()} wells")


In [ ]:
# =======================================================================
# Plate-Map Visualisation of Low-Activity QC code cell
#========================================================================


import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, clear_output

rows, cols = "ABCDEF", range(1, 9)
groups = wide[["Plate", "Timepoint_dpp"]].drop_duplicates().sort_values(
    ["Plate", "Timepoint_dpp"]
)

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()

for ax, (_, g) in zip(axes, groups.iterrows()):

    d = wide[
        (wide["Plate"] == g["Plate"]) &
        (wide["Timepoint_dpp"] == g["Timepoint_dpp"])
    ].set_index("Well")

    for y, row in enumerate(rows):
        for x in cols:
            well = f"{row}{x}"

            if well not in d.index:
                colour = "lightgray"
            elif d.loc[well, "QC_low_activity"] == "fail":
                colour = "red"
            else:
                colour = "green"

            ax.scatter(x, y, s=550, c=colour, edgecolors="black")

    ax.set(
        title=f"Plate {int(g['Plate'])} — {int(g['Timepoint_dpp'])} dpp",
        xticks=list(cols),
        yticks=range(6),
        yticklabels=list(rows),
        xlim=(0.5, 8.5),
        ylim=(5.5, -0.5)
    )

    ax.xaxis.tick_top()
    ax.set_aspect("equal")
    ax.tick_params(length=0)

legend = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="green",
           markeredgecolor="black", markersize=12, label="Pass"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="red",
           markeredgecolor="black", markersize=12, label="Fail"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="lightgray",
           markeredgecolor="black", markersize=12, label="Not in data")
]

fig.legend(handles=legend, loc="lower center", ncol=3)


fig.suptitle(
    "Low-Activity Well Quality Control Results. ActiveElec>=4 & TotalSpikesE >= 50",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0.08, 1, 0.95])
fig.subplots_adjust(hspace=0.3)
plt.savefig("figures/Tier1_qc_plateMap.png", dpi=150)
plt.show()



In [ ]:
# =======================================================================
# Tier-2 quality control — detect abnormal wells among the tier-1 survivors
# =======================================================================
import numpy as np, joblib

MODEL_FILE = os.environ.get("MEA_MODEL", "models/tier2_rf_model.joblib")
THRESHOLD  = 0.50                                   # fail if predicted probability >= this

bundle = joblib.load(MODEL_FILE)
rf, feats, burst = bundle["model"], bundle["feats"], bundle["burst"]

analysable = wide["QC_low_activity"] == "pass"      # tier-2 only judges wells that passed tier-1

# same preprocessing as training: burst flag + fill 0, and match the training columns exactly
w2 = wide.copy()
pb = [c for c in burst if c in w2.columns]
w2["bursted"] = w2[pb[0]].notna().astype(int) if pb else 0
w2[pb] = w2[pb].fillna(0.0)
for c in feats:                                     # ensure every training feature is present
    if c not in w2.columns: w2[c] = 0.0
X_new = w2.loc[analysable, feats].fillna(0.0).to_numpy(float)

# predict fail-probability, then flag
prob = rf.predict_proba(X_new)[:, 1]
wide["tier2_fail_prob"] = pd.NA
wide["tier2_QC"] = "dead_well"                       # tier-1 failures carried through
wide.loc[analysable, "tier2_fail_prob"] = prob.round(3)
wide.loc[analysable, "tier2_QC"] = np.where(prob >= THRESHOLD, "fail", "pass")

wide.to_csv("qc_results_full.csv", index=False)
print(f"Tier-2 among {int(analysable.sum())} analysable wells: "
      f"{int((wide.tier2_QC=='fail').sum())} fail, {int((wide.tier2_QC=='pass').sum())} pass")

In [ ]:
# =======================================================================
# Plate-Map Visualisation of Tier-2 QC
# =======================================================================
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

rows, cols = "ABCDEF", range(1, 9)
groups = wide[["Plate", "Timepoint_dpp"]].drop_duplicates().sort_values(["Plate", "Timepoint_dpp"])
fig, axes = plt.subplots(2, 3, figsize=(13, 7)); axes = axes.flatten()

for ax, (_, g) in zip(axes, groups.iterrows()):
    d = wide[(wide.Plate == g.Plate) & (wide.Timepoint_dpp == g.Timepoint_dpp)].set_index("Well")
    for y, row in enumerate(rows):
        for x in cols:
            well = f"{row}{x}"
            if well not in d.index:                      colour = "lightgray"   # not in data
            elif d.loc[well, "tier2_QC"] == "dead_well":  colour = "dimgray"     # tier-1 fail
            elif d.loc[well, "tier2_QC"] == "fail":       colour = "red"         # tier-2 fail
            else:                                         colour = "green"       # pass
            ax.scatter(x, y, s=550, c=colour, edgecolors="black")
    ax.set(title=f"Plate {int(g.Plate)} — {int(g.Timepoint_dpp)} dpp", xticks=list(cols),
           yticks=range(6), yticklabels=list(rows), xlim=(0.5, 8.5), ylim=(5.5, -0.5))
    ax.xaxis.tick_top(); ax.set_aspect("equal"); ax.tick_params(length=0)

legend = [Line2D([0],[0],marker="o",color="w",markerfacecolor=c,markeredgecolor="black",markersize=12,label=l)
          for c,l in [("green","Pass"),("red","Fail (tier-2)"),("dimgray","Dead well (tier-1)"),("lightgray","Not in data")]]
fig.legend(handles=legend, loc="lower center", ncol=4)
fig.suptitle("Tier-2 Quality Control Results (Random Forest)", fontsize=18, fontweight="bold")
plt.tight_layout(rect=[0, 0.08, 1, 0.95]); fig.subplots_adjust(hspace=0.3)
plt.savefig("figures/tier2_qc_plateMap.png", dpi=150)
plt.show()


In [ ]:
# =======================================================================
# QC data-retention funnel — how many wells survive each stage
# =======================================================================
import numpy as np
import matplotlib.pyplot as plt

raw     = len(wide)
t1_pass = int((wide["QC_low_activity"] == "pass").sum())
t2_fail = int((wide["tier2_QC"] == "fail").sum())
usable  = int((wide["tier2_QC"] == "pass").sum())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={"width_ratios": [1, 1.2]})

# --- left: overall funnel ----------------------------------------------
stages = [("Raw wells", raw, "#9fb4d0"),
          ("Pass tier-1 (active)", t1_pass, "#7bc47f"),
          ("Usable after tier-2", usable, "#2e8b57")
]
drops  = [f"\u2212 {raw - t1_pass} low-activity (tier-1)", f"\u2212 {t2_fail} abnormal (tier-2)"]
for i, (label, n, c) in enumerate(stages):
    y = -i
    a1.barh(y, n, height=0.62, color=c, edgecolor="black")
    a1.text(n + 4, y, f"{n}  ({n/raw:.0%})", va="center", fontsize=11, fontweight="bold")
    a1.text(-6, y, label, va="center", ha="right", fontsize=11)
    if i < len(drops):                                     # what was removed between stages
        a1.text(raw * 1.02, y - 0.5, drops[i], va="center", ha="right",
                fontsize=9.5, color="#b03030", style="italic")
a1.set_xlim(-95, raw * 1.30); a1.set_ylim(-2.6, 0.6); a1.axis("off")
a1.set_title("Wells retained through quality control", fontsize=12.5, fontweight="bold", loc="left")

# --- right: per-recording breakdown ------------------------------------
grp = (wide.assign(rec=lambda d: "Plate " + d.Plate.astype(int).astype(str) + " \u2013 "
                   + d.Timepoint_dpp.astype(int).astype(str) + " dpp")
            .groupby("rec")["tier2_QC"].value_counts().unstack(fill_value=0)
            .reindex(columns=["pass", "fail", "dead_well"], fill_value=0)).iloc[::-1]
left = np.zeros(len(grp))
for col, c, lab in [("pass", "#2e8b57", "usable"), ("fail", "#d84b4b", "tier-2 fail"),
                    ("dead_well", "dimgray", "tier-1 dead")]:
    a2.barh(grp.index, grp[col], left=left, color=c, edgecolor="black", label=lab)
    for y, (v, l) in enumerate(zip(grp[col], left)):       # counts inside the segments
        if v > 0: a2.text(l + v/2, y, str(int(v)), va="center", ha="center",
                          fontsize=9, color="white", fontweight="bold")
    left += grp[col].to_numpy()
a2.set_xlabel("wells")
a2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, fontsize=9.5, frameon=False)
a2.set_title("Per recording", fontsize=12.5, fontweight="bold", loc="left")
for s in ["top", "right"]: a2.spines[s].set_visible(False)

fig.suptitle("QC data-retention funnel", fontsize=15, fontweight="bold")
plt.tight_layout(rect=[0, 0.02, 1, 0.94])
plt.savefig("figures/qc_funnel.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# =======================================================================
# Per-well diagnostic scorecard — "why was this well flagged?"
# Each feature shown as a robust z within the well's OWN recording,
# sorted most-abnormal-first. Call:  well_scorecard(wide, 3, 28, "E4")
# =======================================================================
import numpy as np
import matplotlib.pyplot as plt

META = ["Barcode","Plate","Timepoint_dpp","Well","Control","QC_low_activity",
        "qc_fail","dead_well","tier2_QC","tier2_fail_prob","bursted","consensus_votes"]

def well_scorecard(wide, plate, dpp, well, save=None):
    rec = wide[(wide.Plate == plate) & (wide.Timepoint_dpp == dpp)]
    trg = rec[rec.Well.astype(str).str.upper() == str(well).upper()]
    if trg.empty: print(f"well {well} not found in Plate {plate}, {dpp} dpp"); return None
    trg = trg.iloc[0]
    feats = [c for c in wide.columns if c not in META and pd.api.types.is_numeric_dtype(wide[c])]

    rows = []                                                  # (feature, z, raw, recording z-dist)
    for f in feats:
        v = trg[f]; col = rec[f].dropna()                      # burst feats: bursting wells only
        if pd.isna(v) or len(col) < 5: continue                # skip features undefined for this well
        med, q1, q3 = col.median(), col.quantile(.25), col.quantile(.75)
        scale = (q3 - q1) if (q3 - q1) > 0 else (col.std() or np.nan)
        if not np.isfinite(scale) or scale == 0: continue
        rows.append((f, (v - med) / scale, v, (col - med) / scale))
    rows.sort(key=lambda r: -abs(r[1]))                        # most abnormal first
    n = len(rows)

    fig, ax = plt.subplots(figsize=(9.5, 0.34 * n + 1.6))
    zs = [r[1] for r in rows]
    xlo = min(-3.2, min(zs) * 1.15); xhi = max(3.2, max(zs) * 1.12); pad = (xhi - xlo) * 0.16
    for i, (f, z, v, colz) in enumerate(rows):
        y = n - 1 - i
        ax.boxplot(colz, positions=[y], vert=False, widths=0.55, showfliers=False,
                   boxprops=dict(color="#9aa3ad"), whiskerprops=dict(color="#9aa3ad"),
                   capprops=dict(color="#9aa3ad"), medianprops=dict(color="#5b6572"))
        c = "#d84b4b" if abs(z) > 2.5 else ("#e69d3c" if abs(z) > 1.5 else "#2e8b57")
        ax.scatter(z, y, s=70, color=c, edgecolor="black", zorder=5)
        ax.text(xhi + pad * 0.95, y, f"{v:,.3g}", va="center", ha="right", fontsize=8, color="#333")
    ax.axvline(0, color="#5b6572", lw=1)
    for s_ in (-2.5, 2.5): ax.axvline(s_, color="#d84b4b", lw=1, ls="--", alpha=.6)
    ax.set_xlim(xlo, xhi + pad); ax.set_ylim(-0.7, n - 0.3)
    ax.set_yticks(range(n)); ax.set_yticklabels([r[0] for r in rows[::-1]], fontsize=8.5)
    ax.set_xlabel("robust z within recording   (0 = recording median, dashed = \u00b12.5)")
    ax.text(xhi + pad * 0.95, n - 0.15, "raw value", ha="right", fontsize=8, style="italic", color="#333")
    status = str(trg.get("tier2_QC", "?")).upper(); prob = trg.get("tier2_fail_prob", np.nan)
    ptxt = f", P(fail) = {prob:.2f}" if pd.notna(prob) else ""
    #ax.set_title(f"Well {well} \u2014 Plate {int(plate)}, {int(dpp)} dpp   |   tier-2: {status}{ptxt}",
                 #fontsize=12, fontweight="bold", loc="left")
    for s_ in ["top", "right", "left"]: ax.spines[s_].set_visible(False)
    ax.tick_params(left=False); plt.tight_layout()
    if save: plt.savefig(save, dpi=150, bbox_inches="tight")
    return fig


In [ ]:
# =======================================================================
# Interactive scorecard browser — recording + well dropdowns
# =======================================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

recs = (wide[["Plate", "Timepoint_dpp"]]
        .drop_duplicates()
        .sort_values(["Plate", "Timepoint_dpp"]))

rec_dd = widgets.Dropdown(
    options=[(f"Plate {int(p)}, {int(d)} dpp", (p, d)) for p, d in recs.values],
    description="Recording:", style={"description_width": "initial"},
    layout=widgets.Layout(width="260px"))

well_dd = widgets.Dropdown(
    description="Well:", style={"description_width": "initial"},
    layout=widgets.Layout(width="260px"))

out = widgets.Output()


def _wells_for(plate, dpp):
    """Wells in this recording, most suspicious first."""
    r = wide[(wide.Plate == plate) & (wide.Timepoint_dpp == dpp)]
    if "tier2_fail_prob" in r.columns:
        r = r.sort_values("tier2_fail_prob", ascending=False)
    labels = []
    for _, row in r.iterrows():
        p = row.get("tier2_fail_prob")
        tag = f"  (P={p:.2f})" if pd.notna(p) else ""
        labels.append((f"{row.Well}{tag}", row.Well))
    return labels


def _draw(*_):
    with out:
        clear_output(wait=True)
        if well_dd.value is None:
            return
        plate, dpp = rec_dd.value
        fig_interactive = well_scorecard(wide, plate, dpp, well_dd.value) # Call the modified function
        if fig_interactive:
            display(fig_interactive) # Use display() for explicit output in widgets
            plt.close(fig_interactive) # Close the figure to avoid memory leaks


def _on_rec_change(*_):
    well_dd.unobserve(_draw, names="value")
    well_dd.options = _wells_for(*rec_dd.value)
    well_dd.observe(_draw, names="value")
    _draw()


rec_dd.observe(_on_rec_change, names="value")
well_dd.observe(_draw, names="value")

display(widgets.HBox([rec_dd, well_dd]), out)
_on_rec_change()          # populate and draw the first well

# example: the E4 well from the EDA
#fig_e4 = well_scorecard(wide, 3, 28, "E4", save="well_scorecard_E4.png")